In [1]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
import os
os.environ["UNSLOTH_DISABLE_PATCHING"] = "1"
%pip install transformers==4.56.2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import re
import time
import torch
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from unsloth import FastLanguageModel
import random

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_PATH = "/content/drive/MyDrive/svg_project"
MODEL_PATH = f"{BASE_PATH}/final_model"

TEST_PROMPTS_PATH = f"{BASE_PATH}/test.csv"
SUBMISSION_PATH = f"{BASE_PATH}/submission.csv"
PARTIAL_PATH = f"{BASE_PATH}/submission_partial.csv"

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Loaded model from:", MODEL_PATH)
print(os.listdir(MODEL_PATH))



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded model from: /content/drive/MyDrive/svg_project/final_model
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json']


In [3]:
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask",
    "linearGradient", "radialGradient", "stop",
    "text", "tspan", "title", "desc", "style",
    "pattern", "marker", "filter"
}

def extract_svg(text):
    if not text:
        return ""

    text = text.strip()

    m = SVG_REGEX.search(text)
    if m:
        return m.group(0).strip()

    start = text.lower().find("<svg")
    if start != -1:
        partial = text[start:].strip()

        # trim obvious junk after the svg body if generation rambles
        bad_stops = ["description:", "generate svg", "prompt:", "\n\n\n"]
        low = partial.lower()
        cut_positions = [low.find(s) for s in bad_stops if low.find(s) != -1]
        if cut_positions:
            partial = partial[:min(cut_positions)].strip()

        if "</svg>" not in partial.lower():
            partial += "</svg>"

        return partial

    # NEW: if model outputs drawable fragments like <path .../> without <svg>
    drawable_tags = ["<path", "<rect", "<circle", "<ellipse", "<line", "<polyline", "<polygon", "<g"]
    positions = [text.lower().find(tag) for tag in drawable_tags if text.lower().find(tag) != -1]

    if positions:
        start = min(positions)
        fragment = text[start:].strip()

        bad_stops = ["description:", "generate svg", "prompt:", "\n\n\n"]
        low = fragment.lower()
        cut_positions = [low.find(s) for s in bad_stops if low.find(s) != -1]
        if cut_positions:
            fragment = fragment[:min(cut_positions)].strip()

        return (
            '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
            + fragment +
            "</svg>"
        )

    return ""

def clean_svg(svg):
    if not svg:
        return ""

    svg = svg.strip()

    # normalize namespace prefixes like <ns0:svg> -> <svg>
    svg = re.sub(r"<(/?)ns\d+:svg", r"<\1svg", svg)
    svg = re.sub(r"<(/?)ns\d+:(path|rect|circle|ellipse|line|polyline|polygon|g)\b", r"<\1\2", svg)

    start_match = re.search(r"<(?:ns\d+:)?svg\b", svg, flags=re.IGNORECASE)
    if start_match:
        svg = svg[start_match.start():]

    end_match = list(re.finditer(r"</(?:ns\d+:)?svg>", svg, flags=re.IGNORECASE))
    if end_match:
        svg = svg[:end_match[-1].end()]

    if re.search(r"<svg\b", svg, flags=re.IGNORECASE) and "xmlns=" not in svg:
        svg = re.sub(
            r"<svg\b",
            '<svg xmlns="http://www.w3.org/2000/svg"',
            svg,
            count=1,
            flags=re.IGNORECASE,
        )

    return svg

def normalize_tag(tag):
    return tag.split("}")[-1]

def remove_disallowed_tags(svg_text):
    if not svg_text:
        return ""

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ""

    def strip_bad_children(parent):
        for child in list(parent):
            tag = normalize_tag(child.tag)
            if tag not in ALLOWED_TAGS:
                parent.remove(child)
            else:
                strip_bad_children(child)

    if normalize_tag(root.tag) != "svg":
        return ""

    strip_bad_children(root)
    return ET.tostring(root, encoding="unicode")

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.split("}")[-1] == "svg"
    except ET.ParseError:
        return False

def is_reasonable_svg(svg_text):
    if not svg_text:
        print("fail: empty")
        return False

    if len(svg_text) < 20 or len(svg_text) > 8000:
        print("fail: length", len(svg_text))
        return False

    low = svg_text.lower()
    if "<svg" not in low or "</svg>" not in low:
        print("fail: missing svg tags")
        return False

    bad_phrases = [
        "generate svg",
        "description:",
        "markdown",
        "explanation",
        "here is",
    ]
    for p in bad_phrases:
        if p in low:
            print("fail: bad phrase", p)
            return False

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        print("fail: parse error", e)
        return False

    root_tag = normalize_tag(root.tag)
    if root_tag != "svg":
        print("fail: root tag", root.tag)
        return False

    draw_tags = {
        "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
        "text", "use"
    }

    found = False
    path_count = 0

    for elem in root.iter():
        tag = normalize_tag(elem.tag)

        if tag not in ALLOWED_TAGS:
            print("fail: disallowed tag", tag)
            return False

        if tag == "path":
            path_count += 1

        if tag in draw_tags:
            found = True

    if not found:
        print("fail: no drawable tags")
        return False

    if path_count > 256:
        print("fail: too many paths", path_count)
        return False

    print("pass")
    return True

def fallback_svg(prompt):
    p = prompt.lower()

    def pick_color(text):
        color_map = {
            "blue": "#2563eb",
            "red": "#dc2626",
            "green": "#16a34a",
            "yellow": "#eab308",
            "orange": "#ea580c",
            "purple": "#9333ea",
            "pink": "#db2777",
            "gray": "#6b7280",
            "grey": "#6b7280",
            "black": "#111111",
            "white": "#ffffff",
            "brown": "#8b5a2b",
        }
        for word, val in color_map.items():
            if word in text:
                return val
        return "#111111"

    def has_any(text, words):
        return any(w in text for w in words)

    fg = pick_color(p)
    bg = "#ffffff"

    outlined = has_any(p, ["outlined", "outline", "stroke", "border"])
    filled = has_any(p, ["filled", "solid", "silhouette"]) and not outlined

    stroke = fg
    fill_main = "none" if outlined else fg
    stroke_width = 10 if outlined else 8

    # helper backgrounds
    bg_shapes = []

    if has_any(p, ["white circular background", "within a white circular background", "inside a white circle", "centered within a white circular background"]):
        bg_shapes.append('<circle cx="128" cy="128" r="104" fill="#ffffff" stroke="#d1d5db" stroke-width="4"/>')

    elif has_any(p, ["white background", "plain white background"]):
        bg_shapes.append('<rect x="0" y="0" width="256" height="256" fill="#ffffff"/>')

    if has_any(p, ["square shape", "within a square", "inside a square", "square icon", "box icon"]):
        bg_shapes.append('<rect x="36" y="36" width="184" height="184" rx="20" ry="20" fill="none" stroke="#111111" stroke-width="10"/>')

    shape = ""

    # ========= specific object templates =========

    if "camera" in p:
        shape = (
            f'<rect x="62" y="92" width="132" height="88" rx="16" ry="16" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<circle cx="128" cy="136" r="26" fill="none" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<rect x="88" y="72" width="36" height="20" rx="6" ry="6" fill="{fill_main if not outlined else "none"}" stroke="{stroke}" stroke-width="6"/>'
            f'<circle cx="170" cy="108" r="6" fill="{stroke}"/>'
        )

    elif ("arrow" in p) and ("square" in p or "box" in p or "icon" in p):
        shape = (
            '<rect x="36" y="36" width="184" height="184" rx="20" ry="20" fill="none" stroke="#111111" stroke-width="10"/>'
            f'<path d="M88 164 C88 118, 120 92, 162 92 L176 92 '
            f'M148 64 L188 92 L148 120" fill="none" stroke="{stroke}" stroke-width="12" '
            f'stroke-linecap="round" stroke-linejoin="round"/>'
        )

    elif "arrow" in p:
        shape = (
            f'<path d="M44 128 H182 M144 90 L182 128 L144 166" '
            f'stroke="{stroke}" stroke-width="12" fill="none" '
            f'stroke-linecap="round" stroke-linejoin="round"/>'
        )

    elif "person" in p and "chair" in p:
        shape = (
            f'<circle cx="86" cy="74" r="18" fill="{fg}"/>'
            f'<path d="M86 94 L86 136 L116 156" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
            f'<path d="M104 112 L134 112 L156 152" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
            f'<path d="M82 136 L72 176" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
            f'<path d="M116 156 L116 186" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
            f'<rect x="126" y="112" width="42" height="10" fill="{fg}"/>'
            f'<rect x="162" y="112" width="10" height="54" fill="{fg}"/>'
            f'<rect x="126" y="156" width="42" height="10" fill="{fg}"/>'
        )

    elif "person" in p or "human" in p:
        shape = (
            f'<circle cx="128" cy="72" r="22" fill="{fg}"/>'
            f'<path d="M128 96 L128 156 M96 118 L160 118 M104 204 L128 156 L152 204" '
            f'fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round" stroke-linejoin="round"/>'
        )

    elif "chair" in p:
        shape = (
            f'<rect x="82" y="112" width="62" height="12" fill="{fg}"/>'
            f'<rect x="82" y="112" width="12" height="66" fill="{fg}"/>'
            f'<rect x="144" y="112" width="12" height="66" fill="{fg}"/>'
            f'<rect x="82" y="166" width="74" height="12" fill="{fg}"/>'
            f'<rect x="82" y="72" width="12" height="52" fill="{fg}"/>'
        )

    elif "house" in p or "home" in p:
        shape = (
            f'<polygon points="128,52 198,112 178,112 178,196 78,196 78,112 58,112" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
            f'<rect x="114" y="148" width="28" height="48" fill="none" stroke="{stroke}" stroke-width="8"/>'
        )

    elif "tree" in p:
        shape = (
            f'<circle cx="128" cy="94" r="42" fill="{fg if not outlined else "#86efac"}" stroke="{stroke}" stroke-width="6"/>'
            f'<circle cx="96" cy="112" r="28" fill="{fg if not outlined else "#86efac"}" stroke="{stroke}" stroke-width="6"/>'
            f'<circle cx="160" cy="112" r="28" fill="{fg if not outlined else "#86efac"}" stroke="{stroke}" stroke-width="6"/>'
            f'<rect x="118" y="148" width="20" height="54" fill="#8b5a2b"/>'
        )

    elif "star" in p:
        shape = (
            f'<polygon points="128,42 148,96 206,100 160,136 176,192 128,160 80,192 96,136 50,100 108,96" '
            f'fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
        )

    elif "heart" in p:
        shape = (
            f'<path d="M128 196 L56 124 C38 104 40 72 64 60 C84 50 104 58 116 76 '
            f'C128 58 148 50 168 60 C192 72 194 104 176 124 Z" '
            f'fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
        )

    elif "cloud" in p:
        shape = (
            f'<path d="M82 174 H176 C196 174 210 160 210 142 C210 124 196 110 178 110 '
            f'C172 84 150 66 124 66 C98 66 76 84 70 110 C50 110 34 124 34 144 '
            f'C34 162 48 174 66 174 Z" '
            f'fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
        )

    elif "phone" in p or "smartphone" in p or "mobile" in p:
        shape = (
            f'<rect x="88" y="36" width="80" height="184" rx="16" ry="16" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<circle cx="128" cy="192" r="6" fill="{stroke}"/>'
            f'<rect x="114" y="52" width="28" height="6" rx="3" fill="{stroke}"/>'
        )

    elif "envelope" in p or "mail" in p:
        shape = (
            f'<rect x="48" y="76" width="160" height="104" rx="8" ry="8" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<path d="M48 88 L128 144 L208 88" fill="none" stroke="{stroke}" stroke-width="8" stroke-linejoin="round"/>'
        )

    elif "lock" in p:
        shape = (
            f'<rect x="78" y="112" width="100" height="86" rx="12" ry="12" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<path d="M98 112 V90 C98 70 112 54 128 54 C144 54 158 70 158 90 V112" '
            f'fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
        )

    elif "key" in p:
        shape = (
            f'<circle cx="92" cy="118" r="26" fill="none" stroke="{stroke}" stroke-width="10"/>'
            f'<path d="M118 118 H198 V104 H184 V90 H170 V118" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round" stroke-linejoin="round"/>'
        )

    elif ("play" in p and "button" in p) or ("triangle" in p and "circle" in p):
        shape = (
            f'<circle cx="128" cy="128" r="82" fill="none" stroke="{stroke}" stroke-width="10"/>'
            f'<polygon points="110,92 176,128 110,164" fill="{fg}"/>'
        )

    elif "check" in p or "checkmark" in p or "tick" in p:
        shape = (
            f'<path d="M58 132 L104 176 L198 82" fill="none" stroke="{stroke}" stroke-width="14" '
            f'stroke-linecap="round" stroke-linejoin="round"/>'
        )

    elif "search" in p or "magnifier" in p or "magnifying" in p:
        shape = (
            f'<circle cx="112" cy="112" r="46" fill="none" stroke="{stroke}" stroke-width="10"/>'
            f'<path d="M146 146 L196 196" fill="none" stroke="{stroke}" stroke-width="10" stroke-linecap="round"/>'
        )

    elif "folder" in p:
        shape = (
            f'<path d="M42 94 H98 L114 74 H214 V182 H42 Z" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
        )

    elif "document" in p or "file" in p:
        shape = (
            f'<path d="M82 38 H154 L190 74 V218 H82 Z" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}" stroke-linejoin="round"/>'
            f'<path d="M154 38 V74 H190" fill="none" stroke="{stroke}" stroke-width="8"/>'
            f'<line x1="102" y1="112" x2="170" y2="112" stroke="{stroke}" stroke-width="6"/>'
            f'<line x1="102" y1="136" x2="170" y2="136" stroke="{stroke}" stroke-width="6"/>'
            f'<line x1="102" y1="160" x2="156" y2="160" stroke="{stroke}" stroke-width="6"/>'
        )

    elif "music" in p or "note" in p:
        shape = (
            f'<path d="M154 58 V152 C154 170 140 182 122 182 C106 182 94 172 94 158 '
            f'C94 144 106 134 122 134 C130 134 138 136 144 140 V84 L192 74 V140 '
            f'C192 158 178 170 162 170 C146 170 134 160 134 146 C134 132 146 122 162 122 '
            f'C170 122 178 124 184 128 V66 Z" fill="{fg}"/>'
        )

    elif "sun" in p:
        rays = []
        for x1, y1, x2, y2 in [
            (128, 26, 128, 52), (128, 204, 128, 230),
            (26, 128, 52, 128), (204, 128, 230, 128),
            (54, 54, 72, 72), (184, 184, 202, 202),
            (184, 72, 202, 54), (54, 202, 72, 184),
        ]:
            rays.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{stroke}" stroke-width="8" stroke-linecap="round"/>')
        shape = f'<circle cx="128" cy="128" r="42" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>' + "".join(rays)

    elif "moon" in p:
        shape = (
            f'<path d="M164 52 C128 60 102 92 102 130 C102 168 128 198 166 204 '
            f'C150 214 132 220 114 220 C68 220 32 182 32 136 C32 90 68 52 114 52 '
            f'C132 52 148 52 164 52 Z" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
        )

    elif "mountain" in p:
        shape = f'<polygon points="40,200 100,120 140,170 190,90 216,200" fill="{fg}"/>'

    elif "ladder" in p:
        shape = (
            f'<line x1="80" y1="40" x2="80" y2="216" stroke="{stroke}" stroke-width="10"/>'
            f'<line x1="176" y1="40" x2="176" y2="216" stroke="{stroke}" stroke-width="10"/>'
            f'<line x1="80" y1="70" x2="176" y2="70" stroke="{stroke}" stroke-width="8"/>'
            f'<line x1="80" y1="110" x2="176" y2="110" stroke="{stroke}" stroke-width="8"/>'
            f'<line x1="80" y1="150" x2="176" y2="150" stroke="{stroke}" stroke-width="8"/>'
            f'<line x1="80" y1="190" x2="176" y2="190" stroke="{stroke}" stroke-width="8"/>'
        )

    elif "circle" in p or "round" in p or "ball" in p or "dot" in p:
        shape = f'<circle cx="128" cy="128" r="72" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'

    elif "triangle" in p:
        shape = f'<polygon points="128,40 208,208 48,208" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'

    elif "square" in p or "box" in p or "window" in p:
        shape = f'<rect x="56" y="56" width="144" height="144" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'

    elif "line" in p or "bar" in p:
        shape = f'<line x1="40" y1="128" x2="216" y2="128" stroke="{stroke}" stroke-width="12"/>'

    else:
        # generic but less boring than a single rectangle
        shape = (
            f'<rect x="64" y="64" width="128" height="128" rx="18" ry="18" fill="{fill_main}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
            f'<circle cx="128" cy="128" r="28" fill="none" stroke="{stroke}" stroke-width="8"/>'
        )

    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        + "".join(bg_shapes) +
        shape +
        '</svg>'
    )

def _run_generation(input_text, max_new_tokens=256, do_sample=False):
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

    if do_sample:
        gen_kwargs["temperature"] = 0.7
        gen_kwargs["top_p"] = 0.9

    with torch.no_grad():
        output_ids = model.generate(**gen_kwargs)

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return decoded

def generate_svg(prompt, max_new_tokens=256, debug=False):
    input_text = (
        "Generate only valid SVG code.\n"
        "Return exactly one complete <svg>...</svg> element.\n"
        "Do not include explanations or markdown.\n"
        "Use simple visible SVG shapes.\n"
        f"Description: {prompt}\n"
        "SVG:\n"
    )

    # first try: greedy
    decoded = _run_generation(input_text, max_new_tokens=max_new_tokens, do_sample=False)

    svg = extract_svg(decoded)
    svg = clean_svg(svg)
    svg = remove_disallowed_tags(svg)
    svg = clean_svg(svg)

    # NEW: retry once with sampling if first try is empty/invalid
    if not svg or not is_valid_svg(svg):
        decoded2 = _run_generation(input_text, max_new_tokens=max_new_tokens, do_sample=True)
        svg2 = extract_svg(decoded2)
        svg2 = clean_svg(svg2)
        svg2 = remove_disallowed_tags(svg2)
        svg2 = clean_svg(svg2)

        if svg2 and is_valid_svg(svg2):
            decoded = decoded2
            svg = svg2

    # NEW: final safety net before returning
    if not svg or not is_valid_svg(svg) or not is_reasonable_svg(svg):
        svg = fallback_svg(prompt)

    if debug:
        print("PROMPT:", prompt)
        print("RAW DECODED:", decoded[:1000] if decoded else "[EMPTY]")
        print("EXTRACTED SVG:", svg[:500] if svg else "[EMPTY]")
        print("VALID:", is_valid_svg(svg))
        print("REASONABLE:", is_reasonable_svg(svg))

    return svg

In [4]:
test_df = pd.read_csv(TEST_PROMPTS_PATH)

for i in range(5):
    prompt = test_df.iloc[i]["prompt"]
    svg = generate_svg(prompt, debug=True)
    print("=" * 100)

pass
PROMPT: firewood stack cut logs wood with leaf illustration.
RAW DECODED: <svg width="100" height="100" viewBox="0 0 100 100">
  <rect x="25" y="30" width="40" height="60" fill="#e79c8a"/>
  <circle cx="35" cy="45" r="5" fill="#f0d9b5"/>
  <path d="M30 40 L35 45 L30 50 Z" fill="#ffcc80"/>
</svg>
EXTRACTED SVG: <svg xmlns="http://www.w3.org/2000/svg" xmlns:ns0="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">
  <rect x="25" y="30" width="40" height="60" fill="#e79c8a" />
  <circle cx="35" cy="45" r="5" fill="#f0d9b5" />
  <path d="M30 40 L35 45 L30 50 Z" fill="#ffcc80" />
</svg>
VALID: True
pass
REASONABLE: True
pass
PROMPT: The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.
RAW DECODED: <svg width="100" height="200" viewBox="0 0 100 200">
  <rect x="50" y="30" width="40" height="60" fill="#808080"/>
  <rect x="70" y="30" width="30" height="60" fill="#909090"/>
  <rect x="90" y="30" width="

In [5]:
test_prompts = [
    "A stylized icon depicting a curved arrow within a square shape.",
    "The image features a simple, blue outlined camera icon centered within a white circular background.",
    "The image contains black geometric shapes against a white background, forming an abstract representation of a person sitting on a chair.",
    "A yellow star centered on a plain background.",
    "A red heart icon.",
    "A green tree with a brown trunk.",
    "A magnifying glass search icon.",
    "A document icon with lines.",
    "A music note symbol.",
]

for p in test_prompts:
    print("=" * 100)
    print("PROMPT:", p)
    print(fallback_svg(p))

PROMPT: A stylized icon depicting a curved arrow within a square shape.
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="36" y="36" width="184" height="184" rx="20" ry="20" fill="none" stroke="#111111" stroke-width="10"/><rect x="36" y="36" width="184" height="184" rx="20" ry="20" fill="none" stroke="#111111" stroke-width="10"/><path d="M88 164 C88 118, 120 92, 162 92 L176 92 M148 64 L188 92 L148 120" fill="none" stroke="#111111" stroke-width="12" stroke-linecap="round" stroke-linejoin="round"/></svg>
PROMPT: The image features a simple, blue outlined camera icon centered within a white circular background.
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="104" fill="#ffffff" stroke="#d1d5db" stroke-width="4"/><rect x="62" y="92" width="132" height="88" rx="16" ry="16" fill="none" stroke="#2563eb" stroke-width="10"/><circle cx="128" cy="136" r="26" fill="none" stroke="#2563e

In [6]:
test_df = pd.read_csv(TEST_PROMPTS_PATH)

for i in range(5):
    prompt = test_df.iloc[i]["prompt"]
    svg = generate_svg(prompt, debug=True)
    print("=" * 100)

pass
PROMPT: firewood stack cut logs wood with leaf illustration.
RAW DECODED: <svg width="100" height="100" viewBox="0 0 100 100">
  <rect x="25" y="30" width="40" height="60" fill="#e79c8a"/>
  <circle cx="35" cy="45" r="5" fill="#f0d9b5"/>
  <path d="M30 40 L35 45 L30 50 Z" fill="#ffcc80"/>
</svg>
EXTRACTED SVG: <svg xmlns="http://www.w3.org/2000/svg" xmlns:ns0="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">
  <rect x="25" y="30" width="40" height="60" fill="#e79c8a" />
  <circle cx="35" cy="45" r="5" fill="#f0d9b5" />
  <path d="M30 40 L35 45 L30 50 Z" fill="#ffcc80" />
</svg>
VALID: True
pass
REASONABLE: True
pass
PROMPT: The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.
RAW DECODED: <svg width="100" height="200" viewBox="0 0 100 200">
  <rect x="50" y="30" width="40" height="60" fill="#808080"/>
  <rect x="70" y="30" width="30" height="60" fill="#909090"/>
  <rect x="90" y="30" width="

In [7]:
for p in [PARTIAL_PATH, SUBMISSION_PATH]:
    if os.path.exists(p):
        os.remove(p)
        print("Deleted:", p)

Deleted: /content/drive/MyDrive/svg_project/submission_partial.csv


In [ ]:
test_df = pd.read_csv(TEST_PROMPTS_PATH)

if os.path.exists(PARTIAL_PATH):
    existing_df = pd.read_csv(PARTIAL_PATH)
    rows = existing_df.to_dict("records")
    done_ids = set(existing_df["id"])
    print(f"Resuming from {len(rows)} samples...")
else:
    rows = []
    done_ids = set()
    print("Starting fresh...")

remaining_df = test_df[~test_df["id"].isin(done_ids)].reset_index(drop=True)

SAVE_EVERY = 20
invalid_count = 0
fallback_count = 0
t0 = time.time()

for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df)):
    svg = generate_svg(row["prompt"])
    svg = clean_svg(svg)

    if (
        not is_reasonable_svg(svg)
        or len(svg) > 2500
        or svg.count("<path") > 256
    ):
        invalid_count += 1
        fallback_count += 1

        if invalid_count <= 5:
            print("FAILED PROMPT:", row["prompt"])
            print("RAW SVG:", svg[:500] if svg else "[EMPTY]")
            print("-" * 80)

        svg = fallback_svg(row["prompt"])

    rows.append({"id": row["id"], "svg": svg})

    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(PARTIAL_PATH, index=False)

sub_df = pd.DataFrame(rows)[["id", "svg"]]
sub_df = test_df[["id"]].merge(sub_df, on="id", how="left")

assert list(sub_df.columns) == ["id", "svg"]
assert len(sub_df) == len(test_df)
assert list(sub_df["id"]) == list(test_df["id"])

sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Total Rows: {len(sub_df)}")
print(f"Invalid count: {invalid_count}")
print(f"Fallback count: {fallback_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

check_df = pd.read_csv(SUBMISSION_PATH)

print(check_df.shape)
print(check_df.columns.tolist())
print(check_df["id"].nunique(), "unique ids")
print((check_df["svg"].astype(str).str.len() == 0).sum(), "empty svg rows")
print(check_df.head())

Starting fresh...


  0%|          | 1/1000 [00:07<2:00:39,  7.25s/it]

pass
pass


  0%|          | 2/1000 [00:20<2:57:23, 10.66s/it]

pass
pass


...

pass
pass


 62%|██████▏   | 617/1000 [2:17:05<1:50:54, 17.37s/it]

pass
pass


 62%|██████▏   | 618/1000 [2:17:12<1:30:26, 14.21s/it]

pass
pass
